# 03. 영양소 기준 산출

**목적**: 개인 신체 정보 + 건강 그룹 기반으로 권장 영양소 기준 산출

**근거**: 한국인 영양소 섭취기준(2025), 대한당뇨병학회, 대한고혈압학회 진료지침

**입력**: 나이, 성별, 체중, 신장, 건강 그룹  
**출력**: 권장 칼로리, 단백질, 탄수화물, 지방, 식단 플랜

## 0. 라이브러리 로드

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import koreanize_matplotlib
import warnings

warnings.filterwarnings('ignore')

## 1. 그룹별 식단 플랜 매핑

In [ ]:
# 그룹 → 식단 플랜 매핑
GROUP_TO_MEAL_PLAN = {
    '정상군':           'Balanced Diet',
    '고혈압군':          'Low-Sodium Diet',
    '혈당이상군':         'Low-Carb Diet',
    '비만군':            'Low-Calorie Diet',
    '고혈압+혈당이상군':   'Low-Carb Low-Sodium Diet',
    '고혈압+비만군':      'Low-Calorie Low-Sodium Diet',
    '혈당이상+비만군':    'Low-Carb Low-Calorie Diet',
    '복합위험군':         'Therapeutic Diet',
}

print('=== 그룹별 식단 플랜 매핑 ===')
for group, plan in GROUP_TO_MEAL_PLAN.items():
    print(f'  {group:20s} → {plan}')

## 2. 영양소 기준 산출 함수

**기본 공식** (한국인 영양소 섭취기준 2025 기반)
- 권장 칼로리 = 체중(kg) × 칼로리 계수 (kcal/kg)
- 권장 단백질 = 체중(kg) × 단백질 계수 (g/kg)
- 권장 탄수화물 = 총 칼로리 × 탄수화물 비율 / 4 (kcal/g)
- 권장 지방 = 총 칼로리 × 지방 비율 / 9 (kcal/g)

**그룹별 조정 계수**
- 정상군: 칼로리 30, 탄수화물 55%, 지방 25%
- 고혈압군: 칼로리 30, 탄수화물 55%, 지방 25% (나트륨 2,000mg 이하)
- 혈당이상군: 칼로리 28, 탄수화물 45%, 지방 25% (탄수화물 제한)
- 비만군: 칼로리 25, 탄수화물 50%, 지방 25% (칼로리 제한)
- 복합위험군: 칼로리 25, 탄수화물 45%, 지방 20% (복합 제한)

In [ ]:
# 그룹별 조정 계수
GROUP_COEFFICIENTS = {
    '정상군':           {'kcal_per_kg': 30, 'protein_per_kg': 0.9, 'carb_ratio': 0.55, 'fat_ratio': 0.25},
    '고혈압군':          {'kcal_per_kg': 30, 'protein_per_kg': 0.9, 'carb_ratio': 0.55, 'fat_ratio': 0.25},
    '혈당이상군':         {'kcal_per_kg': 28, 'protein_per_kg': 1.0, 'carb_ratio': 0.45, 'fat_ratio': 0.25},
    '비만군':            {'kcal_per_kg': 25, 'protein_per_kg': 1.0, 'carb_ratio': 0.50, 'fat_ratio': 0.25},
    '고혈압+혈당이상군':   {'kcal_per_kg': 28, 'protein_per_kg': 1.0, 'carb_ratio': 0.45, 'fat_ratio': 0.25},
    '고혈압+비만군':      {'kcal_per_kg': 25, 'protein_per_kg': 1.0, 'carb_ratio': 0.50, 'fat_ratio': 0.20},
    '혈당이상+비만군':    {'kcal_per_kg': 25, 'protein_per_kg': 1.1, 'carb_ratio': 0.45, 'fat_ratio': 0.20},
    '복합위험군':         {'kcal_per_kg': 25, 'protein_per_kg': 1.1, 'carb_ratio': 0.45, 'fat_ratio': 0.20},
}

def calculate_nutrient(age, gender, height, weight, group):
    """
    개인 신체 정보와 건강 그룹을 입력받아 권장 영양소 기준을 산출합니다.

    Parameters:
        age    : 나이
        gender : 성별 (1=남성, 2=여성)
        height : 신장 (cm)
        weight : 체중 (kg)
        group  : 건강 그룹

    Returns:
        dict: 권장 칼로리, 단백질, 탄수화물, 지방, 식단 플랜
    """
    coef = GROUP_COEFFICIENTS.get(group, GROUP_COEFFICIENTS['정상군'])

    # 칼로리 산출
    calories = round(weight * coef['kcal_per_kg'])

    # 나이/성별 보정 (65세 이상 10% 감소, 여성 10% 감소)
    if age >= 65:
        calories = round(calories * 0.9)
    if gender == 2:
        calories = round(calories * 0.9)

    # 영양소 산출
    protein = round(weight * coef['protein_per_kg'], 1)
    carbs   = round(calories * coef['carb_ratio'] / 4, 1)
    fats    = round(calories * coef['fat_ratio'] / 9, 1)

    meal_plan = GROUP_TO_MEAL_PLAN.get(group, 'Balanced Diet')

    return {
        'group':     group,
        'meal_plan': meal_plan,
        'calories':  calories,
        'protein':   protein,
        'carbs':     carbs,
        'fats':      fats,
    }

print('영양소 기준 산출 함수 정의 완료')

## 3. 테스트

In [ ]:
test_cases = [
    {'age': 45, 'gender': 1, 'height': 175, 'weight': 70,  'group': '정상군'},
    {'age': 55, 'gender': 2, 'height': 160, 'weight': 75,  'group': '고혈압+비만군'},
    {'age': 50, 'gender': 1, 'height': 170, 'weight': 80,  'group': '혈당이상군'},
    {'age': 68, 'gender': 2, 'height': 155, 'weight': 65,  'group': '복합위험군'},
]

results = []
for case in test_cases:
    result = calculate_nutrient(**case)
    results.append(result)
    print(f'[{case["age"]}세 {"남" if case["gender"]==1 else "여"} {case["weight"]}kg]')
    print(f'  그룹      : {result["group"]}')
    print(f'  식단 플랜 : {result["meal_plan"]}')
    print(f'  칼로리    : {result["calories"]} kcal')
    print(f'  단백질    : {result["protein"]} g')
    print(f'  탄수화물  : {result["carbs"]} g')
    print(f'  지방      : {result["fats"]} g')
    print()

## 4. 그룹별 영양소 기준 시각화

In [ ]:
# 70kg 남성 45세 기준으로 그룹별 영양소 비교
group_results = []
for group in GROUP_TO_MEAL_PLAN.keys():
    r = calculate_nutrient(age=45, gender=1, height=175, weight=70, group=group)
    group_results.append(r)

df_result = pd.DataFrame(group_results).set_index('group')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(['calories', 'protein', 'carbs', 'fats']):
    df_result[col].plot(kind='barh', ax=axes[i], color='#3498DB', edgecolor='white')
    axes[i].set_title(col, fontsize=12, fontweight='bold')
    for bar in axes[i].patches:
        axes[i].text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                     f'{bar.get_width():.0f}', va='center', fontsize=9)

plt.suptitle('그룹별 권장 영양소 기준 (70kg 남성 45세 기준)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. LLM 프롬프트 구성 예시

In [ ]:
def build_prompt(user_info, nutrient_result):
    """
    LLM에 전달할 프롬프트를 구성합니다.
    """
    prompt = f"""
[사용자 정보]
나이: {user_info['age']}세 / 성별: {'남성' if user_info['gender']==1 else '여성'}
신장: {user_info['height']}cm / 체중: {user_info['weight']}kg
건강 그룹: {nutrient_result['group']}

[영양소 기준]
식단 플랜: {nutrient_result['meal_plan']}
권장 칼로리: {nutrient_result['calories']} kcal
권장 단백질: {nutrient_result['protein']} g
권장 탄수화물: {nutrient_result['carbs']} g
권장 지방: {nutrient_result['fats']} g

위 정보를 바탕으로 아침/점심/저녁 한국인 식단을 구성해줘.
권장 식품과 제한 식품도 함께 제시해줘.
"""
    return prompt

# 테스트
user_info = {'age': 45, 'gender': 1, 'height': 175, 'weight': 70}
nutrient  = calculate_nutrient(group='혈당이상군', **user_info)
print(build_prompt(user_info, nutrient))

## 6. 결과 요약

In [ ]:
print('=' * 60)
print('영양소 기준 산출 결과 요약')
print('=' * 60)
print('근거: 한국인 영양소 섭취기준(2025)')
print('      대한당뇨병학회 진료지침(2023)')
print('      대한고혈압학회 진료지침(2022)')
print()
print(df_result[['meal_plan', 'calories', 'protein', 'carbs', 'fats']].to_string())
print()
print('→ 다음 단계: RAG 문서 검색 + LLM 프롬프트 구성')
print('=' * 60)